## Verificación de la estructura del dataset

Antes de usar RF-DETR, vamos a ver qué carpetas tenemos actualmente en el dataset de parches (versión original), para comprobar si coincide con lo que RF-DETR necesita

In [1]:
import os

ruta = "/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches/originales"

for carpeta in os.listdir(ruta):
    print(carpeta)

dataset_parches_originales.yaml
labels
images


## Verificación de splits (train/val/test)

Vamos a ver qué carpetas hay dentro de `images` y `labels`, para saber si usamos 
"val" o "valid" (RF-DETR espera "valid").

In [2]:
print("Dentro de images:")
for carpeta in os.listdir(ruta + "/images"):
    print(" -", carpeta)

print("\nDentro de labels:")
for carpeta in os.listdir(ruta + "/labels"):
    print(" -", carpeta)

Dentro de images:
 - val
 - test
 - train

Dentro de labels:
 - val
 - test.cache
 - train.cache
 - test
 - val.cache
 - train


## Crear copia completa del dataset para RF-DETR (original y denoised)

Se copia  todo el dataset de parches, tanto la versión original como la denoised, a carpetas nuevas dedicadas a RF-DETR. Así podemos modificarlas (renombrar "val" a 
"valid", etc.) sin tocar el dataset original que usan YOLO y RT-DETR.

In [3]:
import shutil

# Original
origen_original = "/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches/originales"
destino_original = "/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/originales"
shutil.copytree(origen_original, destino_original)
print("Copia 'originales' creada en:", destino_original)

# Denoised
origen_denoised = "/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches/denoised"
destino_denoised = "/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/denoised"
shutil.copytree(origen_denoised, destino_denoised)
print("Copia 'denoised' creada en:", destino_denoised)

Copia 'originales' creada en: /home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/originales
Copia 'denoised' creada en: /home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/denoised


## Ajustes necesarios para RF_DTR


## Renombrar "val" a "valid" (en la copia para RF-DETR)

RF-DETR espera que la carpeta de validación se llame "valid", no "val". Como esto es una copia independiente, se puede  renombrar sin ningún riesgo para el dataset original.

In [4]:
import os

base = "/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR"

for version in ["originales", "denoised"]:
    for tipo in ["images", "labels"]:
        ruta_val = f"{base}/{version}/{tipo}/val"
        ruta_valid = f"{base}/{version}/{tipo}/valid"
        os.rename(ruta_val, ruta_valid)
        print(f"Renombrado: {ruta_val} -> {ruta_valid}")

Renombrado: /home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/originales/images/val -> /home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/originales/images/valid
Renombrado: /home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/originales/labels/val -> /home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/originales/labels/valid
Renombrado: /home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/denoised/images/val -> /home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/denoised/images/valid
Renombrado: /home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/denoised/labels/val -> /home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/denoised/labels/valid


## Revisar el archivo .yaml actual

se revisa  qué contiene el archivo .yaml que se copió, para saber si necesita ajustes para RF-DETR (que espera un archivo llamado "data.yaml").

In [5]:
ruta_yaml = "/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/originales/dataset_parches_originales.yaml"

with open(ruta_yaml) as f:
    contenido = f.read()

print(contenido)


path: /home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches/originales
train: images/train
val: images/val
test: images/test

names:
  0: kanji





## Crear data.yaml para RF-DETR (original y denoised)

Creamos el archivo "data.yaml" que RF-DETR necesita, con la ruta actualizada a la copia nueva y "val" cambiado a "valid".

In [6]:
contenido_yaml = """path: {ruta}
train: images/train
valid: images/valid
test: images/test

names:
  0: kanji
"""

# Original
ruta_original = "/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/originales"
with open(f"{ruta_original}/data.yaml", "w") as f:
    f.write(contenido_yaml.format(ruta=ruta_original))
print("Creado:", f"{ruta_original}/data.yaml")

# Denoised
ruta_denoised = "/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/denoised"
with open(f"{ruta_denoised}/data.yaml", "w") as f:
    f.write(contenido_yaml.format(ruta=ruta_denoised))
print("Creado:", f"{ruta_denoised}/data.yaml")

Creado: /home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/originales/data.yaml
Creado: /home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/denoised/data.yaml


## Eliminar el yaml antiguo (solo en la copia para RF-DETR)

Se elimina  el archivo .yaml de YOLO que se copió, ya que no lo necesita RF-DETR y podría generar confusión. El archivo original (en dataset_kanji_parches/) no se ve 
afectado.

In [7]:
import os

os.remove("/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/originales/dataset_parches_originales.yaml")
os.remove("/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/denoised/dataset_parches_denoised.yaml")

print("Archivos eliminados correctamente")

Archivos eliminados correctamente


## Verificación final de la estructura para RF-DETR

Se comprueba  que la copia tiene todo lo que RF-DETR necesita: data.yaml, y las carpetas images/labels para train, valid y test, con archivos dentro.

In [8]:
import os

for version in ["originales", "denoised"]:
    print(f"=== {version} ===")
    base = f"/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/{version}"
    
    # Verificar data.yaml
    yaml_existe = os.path.exists(f"{base}/data.yaml")
    print(f"data.yaml existe: {yaml_existe}")
    
    # Verificar carpetas y contar archivos
    for split in ["train", "valid", "test"]:
        img_dir = f"{base}/images/{split}"
        lbl_dir = f"{base}/labels/{split}"
        
        n_img = len([f for f in os.listdir(img_dir) if f.endswith(".png")])
        n_lbl = len([f for f in os.listdir(lbl_dir) if f.endswith(".txt")])
        
        print(f"  {split}: {n_img} imágenes, {n_lbl} labels")
    print()

=== originales ===
data.yaml existe: True
  train: 1320 imágenes, 1320 labels
  valid: 240 imágenes, 240 labels
  test: 240 imágenes, 240 labels

=== denoised ===
data.yaml existe: True
  train: 1320 imágenes, 1320 labels
  valid: 240 imágenes, 240 labels
  test: 240 imágenes, 240 labels



## Reorganizar estructura para RF-DETR (denoised)

Repetimos para denoised el mismo proceso ya hecho para "originales": mover de images/labels + split, a split + images/labels, y limpiar carpetas vacías.

In [19]:
base_v = "/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/denoised"

for split in ["train", "valid", "test"]:
    os.makedirs(f"{base_v}/{split}/images", exist_ok=True)
    os.makedirs(f"{base_v}/{split}/labels", exist_ok=True)
    
    for archivo in os.listdir(f"{base_v}/images/{split}"):
        shutil.move(f"{base_v}/images/{split}/{archivo}", f"{base_v}/{split}/images/{archivo}")
    for archivo in os.listdir(f"{base_v}/labels/{split}"):
        shutil.move(f"{base_v}/labels/{split}/{archivo}", f"{base_v}/{split}/labels/{archivo}")

shutil.rmtree(f"{base_v}/images")
shutil.rmtree(f"{base_v}/labels")

print("denoised: reorganización completada")

denoised: reorganización completada



## Verificación final de ambas estructuras

Se confirma  que tanto "originales" como "denoised" quedaron con la misma estructura: 
data.yaml + carpetas train/valid/test, cada una con images/ y labels/ dentro.

In [ ]:
for version in ["originales", "denoised"]:
    base_v = f"/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/{version}"
    print(f"=== {version} ===")
    print("Carpetas:", sorted(os.listdir(base_v)))
    print("Dentro de train:", sorted(os.listdir(f"{base_v}/train")))
    print()

=== originales ===
Carpetas: ['data.yaml', 'test', 'train', 'valid']
Dentro de train: ['images', 'labels']

=== denoised ===
Carpetas: ['data.yaml', 'test', 'train', 'valid']
Dentro de train: ['images', 'labels']



## Problema: incompatibilidad de PyTorch con la GPU (GTX 1080 Ti)

Al intentar entrenar RF-DETR, apareció el error "CUDA error: no kernel image is 
available for execution on the device".

**Causa:** rfdetr instaló automáticamente PyTorch 2.12 (con soporte CUDA para 
arquitecturas sm_75 en adelante), pero las GPUs del servidor son GTX 1080 Ti 
(arquitectura Pascal, sm_61), que ya no está soportada por esa versión de PyTorch.

**Solución:** instalar PyTorch 2.5.1 (la misma versión que funciona correctamente 
en entorno_tfm para YOLO y RT-DETR, que sí soporta sm_61).

## Prueba: cargar el dataset con RF-DETR (sin entrenar)

Se intenta  iniciar un entrenamiento muy corto (1 época) con el dataset de parches original, solo para confirmar que RF-DETR reconoce la estructura correctamente. Si 
funciona, lo detenemos — no necesitamos completar esta época de prueba.

In [4]:
from rfdetr import RFDETRSmall

model = RFDETRSmall()

model.train(
    dataset_dir="/home/salvarado/TFM/Dataset_preparados/dataset_kanji_parches_RFDETR/originales",
    epochs=1,
    batch_size=2,
    grad_accum_steps=4,
    lr=1e-4,
    early_stopping_patience=10,
    seed=42,
    output_dir="/home/salvarado/TFM/resultados/result_RF_DTR/parches_original_50ep_RF_DTR"
)

[2026-06-16 01:05:53] [INFO] rf-detr - File /home/salvarado/.roboflow/models/rf-detr-small.pth already exists with correct MD5 hash.


[2026-06-16 01:05:53] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-16 01:05:53] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-06-16 01:05:54] [INFO] rf-detr - File /home/salvarado/.roboflow/models/rf-detr-small.pth already exists with correct MD5 hash.


[2026-06-16 01:05:56] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-16 01:05:56] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-06-16 01:05:57] [INFO] rf-detr - File /home/salvarado/.roboflow/models/rf-detr-small.pth already exists with correct MD5 hash.


...

Val — Overall Metrics                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃           mAP            ┃  mAR   ┃         F1 sweep         ┃
┡━━━━━━━━┯━━━━━━━━┯━━━━━━━━╇━━━━━━━━╇━━━━━━━━┯━━━━━━━━┯━━━━━━━━┩
│ 50:95  │   50   │   75   │  @500  │   F1   │  Prec  │ Recall │
├────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│ 0.6042 │ 0.9448 │ 0.7009 │ 0.6968 │ 0.9060 │ 0.9231 │ 0.8895 │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┘

                  Val — Per-class Metrics                  
┏━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ Class ┃ AP 50:95 ┃     AR ┃     F1 ┃ Precision ┃ Recall ┃
┡━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ kanji │   0.6042 │ 0.6968 │ 0.9060 │    0.9231 │ 0.8895 │
└───────┴──────────┴────────┴────────┴───────────┴────────┘

[2026-06-16 01:10:59] [INFO] rf-detr - Best regular mAP saved to /home/salvarado/TFM/resultados/result_RF_DTR/parches_original_50ep_RF_DTR/checkpoint_best_regular.pth (epoch 0)
[2026-06-16 01:10:59] [INFO] rf-detr - Best EMA mAP improved to 0.6052 (epoch 0)


`Trainer.fit` stopped: `max_epochs=1` reached.


[2026-06-16 01:11:04] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.6042, ema=0.6052)
